# 🧬 Comprehensive Guide: FACT Spatial Transcriptomics Framework
## Joint Representation Learning, Trainable Neural Clustering, Spatial Refinement & Adaptive Ensemble Engine

---

### 📌 Notebook Overview
This notebook provides an exhaustive, mathematically rigorous, and code-aligned walkthrough of the **Trainable Clustering Framework for Spatial Transcriptomics** implemented in this repository.

The framework integrates four complementary methodologies:
1. **ACT (Autoencoder-Clustering Technique)**: Couples deep autoencoders with MClust neural clustering optimized via KL-divergence loss.
2. **FACT (Filtered Autoencoder-Clustering Technique)**: Extends ACT with an iterative spot-based random sampling filtration algorithm to remove ubiquitous high-background expression noise.
3. **SCATTER**: Implements spot-based expression filtration and autoencoding.
4. **ENSEMBLE**: An adaptive multi-criteria decision framework that automatically selects the optimal spatial domain predictions per sample based on Average Silhouette Width (ASW), PAS, CHAOS, Moran's I, and Geary's C.

---

### 📑 Table of Contents
1. **Architecture & Framework Overview**
2. **Mathematical Foundations & Core Equations**
   - 2.1 Gene Selection & Ubiquitous Expression Filtration Algorithm
   - 2.2 Feature Normalization & Spatial Neighbor Graph Construction
   - 2.3 Deep Autoencoder & Neural Clustering Layer
   - 2.4 Multi-Phase Optimization & Trainable Loss Functions
   - 2.5 Spatial Post-Processing & Radius-Based Majority Refinement
   - 2.6 Spatial Quality & Clustering Evaluation Metrics
   - 2.7 Adaptive Profile-Distance Ensemble Selector
3. **Source Code Architecture & Module Mapping**
4. **Interactive End-to-End Synthetic Workflow (Executable PyTorch Demonstration)**
5. **Key Takeaways & Summary**

---


## 1. 🏗️ Architecture & Framework Overview

Spatial Transcriptomics (ST) pairs high-dimensional gene expression profiles $X \in \mathbb{R}^{N \times G}$ with physical $2\text{D}$ tissue coordinates $P \in \mathbb{R}^{N \times 2}$ across $N$ spatial spots and $G$ genes.

The key challenge in ST domain identification is that raw gene expression is high-dimensional, noisy, and subject to spatial dropouts, while physical coordinates alone lack transcriptomic specificity. The **FACT framework** solves this via a 5-stage pipeline:

```
┌────────────────────────┐    ┌────────────────────────┐    ┌────────────────────────┐
│ 1. Expression Matrix   │ ──►│ 2. Gene Filtration     │ ──►│ 3. Deep Autoencoder    │
│    (Genes x Spots)     │    │    (Seurat + Ubiquitous)│    │    (Latent Dim d = 32) │
└────────────────────────┘    └────────────────────────┘    └────────────────────────┘
                                                                        │
                                                                        ▼
┌────────────────────────┐    ┌────────────────────────┐    ┌────────────────────────┐
│ 5. Adaptive Ensemble   │ ◄──│ 4. Spatial Refinement  │ ◄──│ 4. Neural Clustering   │
│    (ASW/PAS/CHAOS/etc) │    │    (Majority Voting)   │    │    (MClust + KL Loss)  │
└────────────────────────┘    └────────────────────────┘    └────────────────────────┘
```

### Framework Variants
* **ACT**: Performs Seurat v3 highly variable gene selection, standard log-normalization, z-score scaling, and autoencoder feature learning coupled with MClust neural soft-label alignment.
* **FACT**: Introduces an additional spot-level iterative random sampling loop to identify and mask ubiquitously high-expression background genes before autoencoder training.
* **SCATTER**: Alternative spot-level noise filtration approach targeting uninformative gene expression peaks across spots.
* **ENSEMBLE**: Evaluates model predictions on unsupervised spatial metrics ($\text{ASW}, \text{PAS}, \text{CHAOS}, \text{Moran's } I, \text{Geary's } C$) and selects the optimal prediction profile by minimizing Euclidean distance to an ideal metric target vector.


## 2. 🧮 Mathematical Foundations & Core Equations

### 2.1 Gene Selection & Ubiquitous Expression Filtration Algorithm

#### Standard Dispersion Selection (Seurat v3)
Given raw counts matrix $X \in \mathbb{R}^{N \times G}$, highly variable genes are prioritized using variance-stabilizing dispersion estimation:
$$\text{Mean: } \mu_g = \frac{1}{N} \sum_{i=1}^N X_{i,g}, \quad \text{Variance: } \sigma_g^2 = \frac{1}{N-1} \sum_{i=1}^N (X_{i,g} - \mu_g)^2$$
A loess regression models $\log(\sigma_g^2)$ as a function of $\log(\mu_g)$, and genes with the highest standardized variance are retained ($G' = 30,000$).

#### Ubiquitous Expression Masking (FACT & SCATTER)
Certain genes exhibit uniformly high expression across disparate spatial domains, acting as confounding uninformative background noise. FACT filters these via iterative spot sampling:

$$\text{For iterations } t = 1, 2, \dots, 500:$$
1. Sample a random subset of spot indices: $\mathcal{S}^{(t)} \subset \{1, \dots, N\}$ where $|\mathcal{S}^{(t)}| = 200$.
2. For each sampled spot $s \in \mathcal{S}^{(t)}$, identify its top $k=50$ highest-expressed genes:
   $$\text{Top}_{50}(s) = \operatorname{argsort}_{g \in \{1 \dots G'\}}(X_{s, g})[:50]$$
3. Update global gene occurrence counts:
   $$C(g) \leftarrow C(g) + \sum_{s \in \mathcal{S}^{(t)}} \mathbb{I}\left(g \in \text{Top}_{50}(s)\right)$$
4. Select the top 10 most frequent background genes $\mathcal{B} = \operatorname{argsort}_g(C)[:10]$ and mask them out:
   $$\text{HighlyVariable}(g) \leftarrow \text{False} \quad \forall g \in \mathcal{B}$$

---

### 2.2 Feature Normalization & Spatial Neighbor Graph Construction

#### Normalization Pipeline
1. **Target Sum Scaling**: $\tilde{X}_{i,g} = \frac{X_{i,g}}{\sum_{g'} X_{i,g'}} \times 10^4$
2. **Log Transformation**: $X^{\text{log}}_{i,g} = \log(1 + \tilde{X}_{i,g})$
3. **Z-Score Scaling & Clipping**:
   $$X^{\text{scaled}}_{i,g} = \min\left( \frac{X^{\text{log}}_{i,g} - \mu_g}{\sigma_g}, 10 \right)$$

#### Spatial Euclidean Distance & Restricted Neighbor Sets
Let $P_i = (x_i, y_i) \in \mathbb{R}^2$ be spatial coordinates of spot $i$.
Spatial Euclidean distance matrix $D \in \mathbb{R}^{N \times N}$:
$$D_{i,j} = \| P_i - P_j \|_2 = \sqrt{(x_i - x_j)^2 + (y_i - y_j)^2}$$

For each spot $i$, let $\mathcal{N}_i^{\text{top6}}$ be its 6 nearest spatial neighbors ordered by distance:
$$\mathcal{N}_i^{\text{top6}} = \operatorname{argsort}_{j \ne i} (D_{i,j})[:6]$$
Let $d_{i,\text{min}} = D_{i, \mathcal{N}_i^{\text{top6}}[0]}$ be the distance to the single nearest neighbor. The restricted spatial neighborhood $\mathcal{N}_i^{\text{small}}$ retains only immediate physical neighbors:
$$\mathcal{N}_i^{\text{small}} = \left\{ j \in \mathcal{N}_i^{\text{top6}} \mid D_{i,j} < 1.5 \cdot d_{i,\text{min}} \right\}$$

---

### 2.3 Deep Autoencoder & Neural Clustering Layer

The autoencoder compresses normalized expression $X \in \mathbb{R}^{N \times G'}$ into a dense latent spatial representation $Z \in \mathbb{R}^{N \times d}$ (where $d = 32$) and decodes it back to $\hat{X}$.

#### Layer Specifications
* **Encoder Weight**: $W_1 \in \mathbb{R}^{G' \times d}$
* **Decoder Weight**: $W_2 \in \mathbb{R}^{d \times G'}$
* **Cluster Classifier Weight**: $W_3 \in \mathbb{R}^{d \times K}$ (where $K=7$ spatial domain clusters)

#### Forward Pass Equations
$$\text{Latent Embedding: } Z = X W_1 \quad (Z \in \mathbb{R}^{N \times 32})$$
$$\text{Reconstructed Expression: } \hat{X} = Z W_2 \quad (\hat{X} \in \mathbb{R}^{N \times G'})$$
$$\text{Unnormalized Cluster Logits: } L = Z W_3 \quad (L \in \mathbb{R}^{N \times K})$$
$$\text{Softmax Cluster Assignment: } P_{i,k} = \frac{\exp(L_{i,k})}{\sum_{k'=1}^K \exp(L_{i,k'})}$$

---

### 2.4 Multi-Phase Optimization & Trainable Loss Functions

Model training proceeds across **3 distinct optimization phases** over $T=3000$ epochs:

```
Epoch 0                   Epoch 1000                Epoch 2000               Epoch 3000
   │                         │                         │                        │
   ▼                         ▼                         ▼                        ▼
┌─────────────────────────┐ ┌─────────────────────────┐ ┌────────────────────────┐
│ Phase 1: Feature Warmup │ │ Phase 2: Encoder Freeze │ │ Phase 3: Joint Train   │
│ L = α * L_recon         │ │ L = L_mse(P, Y_mclust)  │ │ L = α*L_recon +        │
│                         │ │ (Freeze W1)             │ │     2*α*L_kl (Unfreeze)│
└─────────────────────────┘ └─────────────────────────┘ └────────────────────────┘
```

#### Phase 1: Reconstruction Warmup ($0 \le t < 1000$)
Objective is pure autoencoding feature compression:
$$\mathcal{L}_{\text{recon}} = \frac{1}{N \cdot G'} \sum_{i=1}^N \sum_{g=1}^{G'} (X_{i,g} - \hat{X}_{i,g})^2$$
$$\mathcal{L}^{(1)} = \alpha \cdot \mathcal{L}_{\text{recon}} \quad (\alpha = 10)$$

#### Phase 2: Soft Label Alignment & Encoder Freeze ($1000 \le t < 2000$)
At $t = 1000$, encoder weights $W_1$ are frozen ($\text{requires\_grad} = \text{False}$).
Every 100 epochs, MClust is applied to latent embedding $Z$, followed by spatial neighbor label smoothing:
$$Y_i^{\text{mclust}} = \operatorname{MClust}(Z)_i$$
For spots whose neighbors share a strong consensus ($\ge 5$ of neighbors share label $y_0$):
$$\text{If } \operatorname{count}(Y_{\mathcal{N}_i^{\text{small}}}^{\text{mclust}} = y_0) > 4 \implies Y_i^{\text{mclust}} \leftarrow y_0$$
Target labels are converted to one-hot targets $Y \in \{0,1\}^{N \times K}$:
$$\mathcal{L}^{(2)} = \text{MSE}\left(P, Y\right) = \frac{1}{N K} \sum_{i=1}^N \sum_{k=1}^K (P_{i,k} - Y_{i,k})^2$$

#### Phase 3: Joint Optimization via KL-Divergence ($t \ge 2000$)
At $t = 2000$, encoder weights $W_1$ are unfrozen ($\text{requires\_grad} = \text{True}$).
Every 50 epochs, MClust labels $Y$ are refreshed with spatial consensus smoothing.
The model is trained end-to-end minimizing combined Reconstruction MSE and KL-Divergence Loss:

$$\mathcal{L}_{\text{KL}}(P \parallel Y) = \frac{1}{N} \sum_{i=1}^N \sum_{k=1}^K P_{i,k} \log \left( \frac{P_{i,k} + \epsilon}{Y_{i,k} + \epsilon} \right)$$
$$\mathcal{L}^{(3)} = \alpha \cdot \mathcal{L}_{\text{recon}} + 2 \cdot \alpha \cdot \mathcal{L}_{\text{KL}}(P \parallel Y)$$

---

### 2.5 Spatial Post-Processing & Radius-Based Majority Refinement

After model optimization, predicted cluster labels $y_i \in \{1, \dots, K\}$ undergo spatial post-processing using radius threshold $R = 50$:

For each spot $i$, collect all spatial neighbors within radius $R$:
$$\mathcal{N}_i(R) = \{ j \in \{1, \dots, N\} \mid D_{i,j} \le R \}$$

Assign the refined cluster label $y_i^{\text{refined}}$ via majority voting:
$$y_i^{\text{refined}} = \operatorname{mode}\left( \{ y_j \mid j \in \mathcal{N}_i(R) \} \right)$$

This eliminates isolated single-spot classification noise while preserving continuous tissue layer boundaries.

---

### 2.6 Spatial Quality & Clustering Evaluation Metrics

#### 1. Adjusted Rand Index (ARI) - Supervised
Given ground truth annotations $G^*$ and predicted clusters $C$:
$$\text{ARI} = \frac{\sum_{ij} \binom{n_{ij}}{2} - \left[ \sum_i \binom{a_i}{2} \sum_j \binom{b_j}{2} \right] / \binom{n}{2}}{\frac{1}{2} \left[ \sum_i \binom{a_i}{2} + \sum_j \binom{b_j}{2} \right] - \left[ \sum_i \binom{a_i}{2} \sum_j \binom{b_j}{2} \right] / \binom{n}{2}}$$

#### 2. CHAOS (Cluster Shape Spatial Dispersion) - Unsupervised
Measures continuous boundary tightness in spatial coordinate space $P$:
$$\text{CHAOS} = \frac{1}{N} \sum_{k=1}^K \sum_{i \in C_k} \min_{j \in C_k, j \ne i} \| P_i - P_j \|_2$$
*(Lower CHAOS indicates compact, non-fragmented spatial domains).*

#### 3. PAS (Position Association Score) - Unsupervised
Quantifies local neighborhood domain continuity in physical space:
$$\text{PAS} = \frac{1}{N} \sum_{i=1}^N \mathbb{I}\left( \sum_{j \in \text{KNN}_{10}(i)} \mathbb{I}(y_j \ne y_i) > 5 \right)$$
*(Lower PAS indicates smoother spatial transitions).*

#### 4. ASW (Average Silhouette Width) - Unsupervised
Evaluates cluster separation in latent embedding space $Z$:
$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}, \quad \text{ASW} = \frac{1}{N} \sum_{i=1}^N s(i)$$
where $a(i)$ is mean intra-cluster distance and $b(i)$ is mean nearest inter-cluster distance.

#### 5. Spatial Autocorrelation (Moran's I & Geary's C)
For top domain marker genes $g$, given spatial spatial weight matrix $W_{ij}$:
$$\text{Moran's } I = \frac{N}{\sum_{i,j} W_{ij}} \frac{\sum_{i,j} W_{ij} (X_{i,g} - \bar{X}_g)(X_{j,g} - \bar{X}_g)}{\sum_i (X_{i,g} - \bar{X}_g)^2}$$
$$\text{Geary's } C = \frac{(N-1)}{2 \sum_{i,j} W_{ij}} \frac{\sum_{i,j} W_{ij} (X_{i,g} - X_{j,g})^2}{\sum_i (X_{i,g} - \bar{X}_g)^2}$$

---

### 2.7 Adaptive Profile-Distance Ensemble Selector

The `Ensemble.py` module evaluates method predictions across 5 key metrics:
$$\mathcal{M} = \{\text{ASW}, \text{MoranI}, \text{PAS}, \text{CHAOS}, \text{GearyC}\}$$
where $\mathcal{M}_{\text{max}} = \{\text{ASW}, \text{MoranI}\}$ (higher is better) and $\mathcal{M}_{\text{min}} = \{\text{PAS}, \text{CHAOS}, \text{GearyC}\}$ (lower is better).

#### Step 1: Benefit Normalization
For each metric $k \in \mathcal{M}$ across candidate methods $m \in \{1, \dots, M\}$:
$$z_{m,k} = \frac{v_{m,k} - \min_{m'} v_{m',k}}{\max_{m'} v_{m',k} - \min_{m'} v_{m',k} + \epsilon}$$
If $k \in \mathcal{M}_{\text{min}}$, convert to benefit orientation:
$$z_{m,k} \leftarrow 1.0 - z_{m,k}$$

#### Step 2: Target Profile Construction
Construct an ideal performance target vector $t \in \mathbb{R}^{|\mathcal{M}|}$:
$$t_k = \begin{cases} \max_m z_{m,k} & \text{if } k \in \mathcal{M}_{\text{max}} \\ \operatorname{median}_m z_{m,k} & \text{if } k \in \mathcal{M}_{\text{min}} \end{cases}$$

#### Step 3: Profile Distance Selection
Select the method $m^*$ minimizing squared Euclidean distance to target vector $t$:
$$m^* = \arg\min_{m \in \{1 \dots M\}} \sum_{k \in \mathcal{M}} (z_{m,k} - t_k)^2$$


## 3. 📂 Source Code Architecture & Module Mapping

The repository structures these algorithms into modular Python scripts inside `methods/` and `Ensemble.py`:

| Mathematical Module | Source Code Location | Key Functions / Classes | Role in Pipeline |
| :--- | :--- | :--- | :--- |
| **Gene Filtration** | [`FACT_main.py`](file:///d:/FYDP/FACTSpatialTranscriptomics/methods/FACT_main.py#L216-L237) | `Counter`, `random.randint` loop | Spot-based random sampling & background gene masking |
| | [`scatter.py`](file:///d:/FYDP/FACTSpatialTranscriptomics/methods/scatter.py) | Spot sampling loop | Spot expression peak filtration |
| **Preprocessing** | [`simple_preprocess.py`](file:///d:/FYDP/FACTSpatialTranscriptomics/methods/simple_preprocess.py) | `preprocess()`, `get_feature()` | Target sum norm, log1p transform, z-score scaling |
| **Distance & Neighbors**| [`ACT_main.py`](file:///d:/FYDP/FACTSpatialTranscriptomics/methods/ACT_main.py#L148-L184) | `ot.dist()`, `d_small` dict | Euclidean spatial matrix & restricted neighbor construction |
| **Autoencoder Model** | [`autoencoder_model_cluster.py`](file:///d:/FYDP/FACTSpatialTranscriptomics/methods/autoencoder_model_cluster.py#L49-L91) | `class Encoder(Module)` | PyTorch Encoder ($W_1$), Decoder ($W_2$), Classifier ($W_3$) |
| **Network Trainer** | [`ACT_Network.py`](file:///d:/FYDP/FACTSpatialTranscriptomics/methods/ACT_Network.py#L24-L471) | `class Embedding_Network` | 3-Phase optimization loop (MSE, MClust freeze, KL Loss) |
| **Clustering & Refine**| [`utils_edit_PCA.py`](file:///d:/FYDP/FACTSpatialTranscriptomics/methods/utils_edit_PCA.py#L54-L154) | `clustering()`, `refine_label()` | R `mclust` integration & radius majority voting refinement |
| **Evaluation Metrics**| [`check.py`](file:///d:/FYDP/FACTSpatialTranscriptomics/methods/check.py) | `compute_CHAOS()`, `compute_PAS()`, `compute_ASW()` | CHAOS, PAS, ASW, Moran's I, Geary's C metric calculation |
| **Ensemble Engine** | [`Ensemble.py`](file:///d:/FYDP/FACTSpatialTranscriptomics/Ensemble.py#L19-L74) | `select_ensemble_method_per_sample()` | Min-Max benefit transformation & target profile selector |


## 4. 💻 Interactive End-to-End Synthetic Workflow

The following executable Python code cells implement the complete FACT pipeline step-by-step using a synthetic spatial transcriptomics dataset.

We will simulate:
1. **Synthetic Spatial Spots**: $N = 400$ spots arranged on a 2D grid with 4 distinct spatial tissue layers.
2. **Gene Expression Matrix**: $G = 300$ genes, including domain-specific marker genes and ubiquitous background noise genes.
3. **Complete Execution**: Gene filtration $\to$ Graph building $\to$ Autoencoder training $\to$ KL loss alignment $\to$ Spatial label refinement $\to$ Metric calculation $\to$ Ensemble decision selection.


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.spatial.distance import cdist
from scipy.stats import mode
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
import random
import matplotlib.pyplot as plt

# Set reproducible seeds
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

print("Initializing Synthetic Spatial Transcriptomics Dataset...")

# 1. Generate 2D spatial coordinates (Grid layout simulating Visium tissue slice)
n_side = 20
x_coords = np.repeat(np.arange(n_side), n_side)
y_coords = np.tile(np.arange(n_side), n_side)
spatial_coords = np.column_stack([x_coords, y_coords])
N = len(spatial_coords) # 400 spots

# 2. Define 4 true synthetic spatial domains (Layer 1-4 based on spatial geometry)
ground_truth = np.zeros(N, dtype=int)
for i, (x, y) in enumerate(spatial_coords):
    if x + y < 14:
        ground_truth[i] = 1 # Top-left domain
    elif x + y < 24:
        ground_truth[i] = 2 # Middle band
    elif x < 12:
        ground_truth[i] = 3 # Bottom-left domain
    else:
        ground_truth[i] = 4 # Right domain

# 3. Generate gene expression profiles G = 300
G = 300
expression_matrix = np.random.poisson(lam=1.5, size=(N, G)).astype(float)

# Inject domain-specific marker gene signals
for spot_idx in range(N):
    domain = ground_truth[spot_idx]
    start_g = (domain - 1) * 25
    end_g = start_g + 25
    expression_matrix[spot_idx, start_g:end_g] += np.random.poisson(lam=8.0, size=25)

# Inject 5 UBIQUITOUS HIGH-EXPRESSION BACKGROUND NOISE GENES across ALL spots
background_genes = [280, 281, 282, 283, 284]
expression_matrix[:, background_genes] += np.random.poisson(lam=25.0, size=(N, 5))

print(f"Generated dataset: {N} spots, {G} genes, 4 Ground Truth Spatial Domains.")

# Visualize Synthetic Ground Truth Domains
plt.figure(figsize=(6, 5))
scatter = plt.scatter(spatial_coords[:, 0], spatial_coords[:, 1], c=ground_truth, cmap='tab10', s=40)
plt.title("Synthetic Spatial Tissue Domains (Ground Truth)")
plt.xlabel("X Coordinate")
plt.ylabel("Y Coordinate")
plt.colorbar(scatter, label="Spatial Domain")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# STEP 1: FACT / SCATTER UBIQUITOUS EXPRESSION FILTRATION & NORMALIZATION
# ==============================================================================

def fact_gene_filtration(counts, num_iterations=200, sample_size=50, top_k=10):
    N_spots, G_genes = counts.shape
    gene_counts = np.zeros(G_genes, dtype=int)
    
    for _ in range(num_iterations):
        sampled_spots = np.random.choice(N_spots, size=min(sample_size, N_spots), replace=False)
        for s in sampled_spots:
            top_genes = np.argsort(counts[s])[::-1][:10]
            gene_counts[top_genes] += 1
            
    masked_genes = np.argsort(gene_counts)[::-1][:top_k]
    variable_mask = np.ones(G_genes, dtype=bool)
    variable_mask[masked_genes] = False
    return variable_mask, masked_genes

# Run FACT Gene Filtration
var_mask, masked_bg_genes = fact_gene_filtration(expression_matrix, num_iterations=200, top_k=5)
print(f"FACT Gene Filtration Masked Ubiquitous Genes: {masked_bg_genes.tolist()}")

# Filter expression matrix
filtered_counts = expression_matrix[:, var_mask]

# Normalization: Target Sum Scaling + Log1p + Z-Score Standard Scaling
total_counts = filtered_counts.sum(axis=1, keepdims=True)
norm_counts = (filtered_counts / total_counts) * 1e4
log_counts = np.log1p(norm_counts)
scaled_features = (log_counts - log_counts.mean(axis=0)) / (log_counts.std(axis=0) + 1e-8)
scaled_features = np.clip(scaled_features, -10, 10)

print(f"Preprocessed Matrix Shape: {scaled_features.shape}")


In [ ]:
# ==============================================================================
# STEP 2: SPATIAL DISTANCE MATRIX & RESTRICTED NEIGHBOR SETS
# ==============================================================================

distance_matrix = cdist(spatial_coords, spatial_coords, metric='euclidean')

d_small = {}
for i in range(N):
    distances_i = distance_matrix[i]
    sorted_indices = np.argsort(distances_i)
    
    top_6 = sorted_indices[1:7]
    min_dist = distances_i[top_6[0]]
    
    restricted_neighbors = [j for j in top_6 if distances_i[j] < 1.5 * min_dist]
    d_small[i] = restricted_neighbors

print(f"Spatial Distance Matrix Calculated: {distance_matrix.shape}")
print(f"Sample Spot 0 Restricted Neighbors: {d_small[0]}")


In [ ]:
# ==============================================================================
# STEP 3: PYTORCH AUTOENCODER & MULTI-PHASE TRAINABLE LOSS OPTIMIZATION
# ==============================================================================

class PyTorchEncoder(nn.Module):
    def __init__(self, in_features, out_features=32, n_clusters=4):
        super(PyTorchEncoder, self).__init__()
        self.weight1 = nn.Parameter(torch.FloatTensor(in_features, out_features))
        self.weight2 = nn.Parameter(torch.FloatTensor(out_features, in_features))
        self.weight3 = nn.Parameter(torch.FloatTensor(out_features, n_clusters))
        
        nn.init.xavier_uniform_(self.weight1)
        nn.init.xavier_uniform_(self.weight2)
        nn.init.xavier_uniform_(self.weight3)

    def forward(self, feat):
        z = torch.mm(feat, self.weight1)     # Latent embedding (N x 32)
        h = torch.mm(z, self.weight2)        # Reconstruction (N x G)
        logits = torch.mm(z, self.weight3)   # Classifier logits (N x K)
        return z, h, logits

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
in_dim = scaled_features.shape[1]
K_clusters = 4
model = PyTorchEncoder(in_features=in_dim, out_features=32, n_clusters=K_clusters).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

X_tensor = torch.FloatTensor(scaled_features).to(device)

print(f"Starting Multi-Phase PyTorch Training on device: {device}...")

epochs = 600
alpha = 10.0
eps = 1e-8

current_target_labels = torch.zeros(N, K_clusters).to(device)

for epoch in range(epochs):
    model.train()
    
    if epoch < 200:
        model.weight1.requires_grad = True
        z, h, logits = model(X_tensor)
        loss_recon = F.mse_loss(X_tensor, h)
        loss = alpha * loss_recon

    elif epoch < 400:
        if epoch == 200:
            model.weight1.requires_grad = False
            
        z, h, logits = model(X_tensor)
        
        if epoch % 50 == 0:
            z_np = z.detach().cpu().numpy()
            from sklearn.cluster import KMeans
            km = KMeans(n_clusters=K_clusters, random_state=42, n_init=5).fit(z_np)
            labels_pseudo = km.labels_
            
            smoothed_labels = labels_pseudo.copy()
            for i in range(N):
                neigh_labs = [labels_pseudo[j] for j in d_small[i]]
                if len(neigh_labs) > 0:
                    most_common = max(set(neigh_labs), key=neigh_labs.count)
                    if neigh_labs.count(most_common) >= len(neigh_labs) / 2:
                        smoothed_labels[i] = most_common
                        
            current_target_labels = F.one_hot(torch.tensor(smoothed_labels).long(), num_classes=K_clusters).float().to(device)
            
        prob_labels = F.softmax(logits, dim=1)
        loss = F.mse_loss(prob_labels, current_target_labels)

    else:
        if epoch == 400:
            model.weight1.requires_grad = True
            
        z, h, logits = model(X_tensor)
        
        if epoch % 20 == 0:
            z_np = z.detach().cpu().numpy()
            from sklearn.cluster import KMeans
            km = KMeans(n_clusters=K_clusters, random_state=42, n_init=5).fit(z_np)
            smoothed_labels = km.labels_
            current_target_labels = F.one_hot(torch.tensor(smoothed_labels).long(), num_classes=K_clusters).float().to(device)
            
        prob_labels = F.softmax(logits, dim=1)
        loss_recon = F.mse_loss(X_tensor, h)
        loss_kl = F.kl_div(prob_labels.clamp_min(eps).log(), current_target_labels.clamp_min(eps), reduction='batchmean')
        
        loss = alpha * loss_recon + 2.0 * alpha * loss_kl

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] | Loss: {loss.item():.4f}")

model.eval()
with torch.no_grad():
    final_latent, _, final_logits = model(X_tensor)
    latent_z = final_latent.cpu().numpy()
    pred_raw = torch.argmax(F.softmax(final_logits, dim=1), dim=1).cpu().numpy()

print("Model Optimization Complete!")


In [ ]:
# ==============================================================================
# STEP 4: RADIUS-BASED MAJORITY VOTING SPATIAL LABEL REFINEMENT
# ==============================================================================

def refine_spatial_labels(coords, labels, radius=2.5):
    dist_mat = cdist(coords, coords, metric='euclidean')
    N_spots = len(labels)
    refined_labels = labels.copy()
    
    for i in range(N_spots):
        neighbors_within_radius = np.where(dist_mat[i] <= radius)[0]
        neighbor_labels = labels[neighbors_within_radius]
        
        vals, counts = np.unique(neighbor_labels, return_counts=True)
        majority_label = vals[np.argmax(counts)]
        refined_labels[i] = majority_label
        
    return refined_labels

pred_refined = refine_spatial_labels(spatial_coords, pred_raw, radius=2.2)

ari_raw = adjusted_rand_score(ground_truth, pred_raw)
ari_refined = adjusted_rand_score(ground_truth, pred_refined)

print(f"ARI Before Spatial Refinement: {ari_raw:.4f}")
print(f"ARI After Spatial Refinement:  {ari_refined:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].scatter(spatial_coords[:, 0], spatial_coords[:, 1], c=ground_truth, cmap='tab10', s=35)
axes[0].set_title("Ground Truth Domains")

axes[1].scatter(spatial_coords[:, 0], spatial_coords[:, 1], c=pred_raw, cmap='tab10', s=35)
axes[1].set_title(f"Unrefined Predictions (ARI = {ari_raw:.3f})")

axes[2].scatter(spatial_coords[:, 0], spatial_coords[:, 1], c=pred_refined, cmap='tab10', s=35)
axes[2].set_title(f"Refined Predictions (ARI = {ari_refined:.3f})")

for ax in axes:
    ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# STEP 5: MULTI-METRIC SPATIAL & CLUSTERING EVALUATION
# ==============================================================================

def compute_chaos(labels, coords):
    from sklearn.preprocessing import StandardScaler
    norm_coords = StandardScaler().fit_transform(coords)
    unique_labels = np.unique(labels)
    cluster_dists = []
    
    for k in unique_labels:
        pts = norm_coords[labels == k]
        if len(pts) <= 2:
            continue
        dmat = cdist(pts, pts)
        np.fill_diagonal(dmat, np.inf)
        min_1nn = dmat.min(axis=1)
        cluster_dists.append(min_1nn.sum())
        
    return np.sum(cluster_dists) / len(labels)

def compute_pas(labels, coords, k=10):
    dmat = cdist(coords, coords)
    np.fill_diagonal(dmat, np.inf)
    N_spots = len(labels)
    diff_count = 0
    
    for i in range(N_spots):
        knn_idx = np.argsort(dmat[i])[:k]
        neighbor_diffs = np.sum(labels[knn_idx] != labels[i])
        if neighbor_diffs > (k / 2):
            diff_count += 1
            
    return diff_count / N_spots

asw = silhouette_score(latent_z, pred_refined)
chaos = compute_chaos(pred_refined, spatial_coords)
pas = compute_pas(pred_refined, spatial_coords)
nmi = normalized_mutual_info_score(ground_truth, pred_refined)

print("=======================================================")
print("          FACT MODEL EVALUATION METRICS                ")
print("=======================================================")
print(f"  * Adjusted Rand Index (ARI) [up]:       {ari_refined:.4f}")
print(f"  * Normalized Mutual Info (NMI) [up]:    {nmi:.4f}")
print(f"  * Average Silhouette Width (ASW) [up]:  {asw:.4f}")
print(f"  * CHAOS Score (Boundary Dispersion) [down]:{chaos:.4f}")
print(f"  * PAS Score (Neighborhood Continuity)[down]:{pas:.4f}")
print("=======================================================")


In [ ]:
# ==============================================================================
# STEP 6: ADAPTIVE PROFILE-DISTANCE ENSEMBLE SELECTOR
# ==============================================================================

df_metrics = pd.DataFrame([
    {"sample": "Sample_01", "method": "ACT",      "ARI": ari_raw,     "ASW": asw-0.02, "PAS": pas+0.03, "CHAOS": chaos+0.04, "MoranI": 0.25, "GearyC": 0.75},
    {"sample": "Sample_01", "method": "FACT",     "ARI": ari_refined, "ASW": asw,      "PAS": pas,      "CHAOS": chaos,      "MoranI": 0.32, "GearyC": 0.68},
    {"sample": "Sample_01", "method": "SCATTER",  "ARI": 0.78,        "ASW": 0.21,     "PAS": 0.08,     "CHAOS": 0.42,       "MoranI": 0.28, "GearyC": 0.72},
    {"sample": "Sample_01", "method": "STAGATE",  "ARI": 0.71,        "ASW": 0.18,     "PAS": 0.12,     "CHAOS": 0.49,       "MoranI": 0.22, "GearyC": 0.78},
])

def select_ensemble_method(df):
    MAXIMIZE = {"ASW", "MoranI"}
    MINIMIZE = {"PAS", "CHAOS", "GearyC"}
    
    Z = {}
    for col in (MAXIMIZE | MINIMIZE):
        vals = df[col].to_numpy(dtype=float)
        mn, mx = vals.min(), vals.max()
        z = (vals - mn) / (mx - mn + 1e-12) if abs(mx - mn) > 1e-12 else np.full_like(vals, 0.5)
        if col in MINIMIZE:
            z = 1.0 - z
        Z[col] = z
    Z_df = pd.DataFrame(Z)
    
    target = {}
    for col in Z_df.columns:
        if col in MAXIMIZE:
            target[col] = Z_df[col].max()
        else:
            target[col] = Z_df[col].median()
    target_vec = np.array([target[col] for col in Z_df.columns])
    
    Z_mat = Z_df.to_numpy()
    d2 = np.sum((Z_mat - target_vec[None, :]) ** 2, axis=1)
    best_idx = np.argmin(d2)
    
    return df.iloc[best_idx]["method"], df.iloc[best_idx]["ARI"], d2

selected_method, selected_ari, dist_scores = select_ensemble_method(df_metrics)

df_metrics["Distance_To_Target"] = dist_scores
print("=========================================================================")
print("               MULTI-CRITERIA ENSEMBLE DECISION MATRIX                   ")
print("=========================================================================")
print(df_metrics[["method", "ARI", "ASW", "PAS", "CHAOS", "Distance_To_Target"]].to_string(index=False))
print("=========================================================================")
print(f"ENSEMBLE SELECTION FOR SAMPLE_01: Method '{selected_method}' (Selected ARI: {selected_ari:.4f})")
print("=========================================================================")


## 5. 💡 Key Takeaways & Summary

### 📌 Summary of Framework Strengths
1. **Targeted Noise Removal**: FACT's spot-level iterative random sampling removes ubiquitous non-informative genes, preventing loss of domain boundary resolution during autoencoding.
2. **Trainable Loss Coupling**: Instead of decoupled representation learning and post-hoc clustering, FACT aligns PyTorch network logits directly with spatially-smoothed MClust pseudo-labels via joint KL-divergence loss.
3. **Spatial Continuity**: Post-processing radius majority voting replaces noisy spot anomalies with local neighborhood consensus.
4. **Adaptive Multi-Metric Selection**: The profile-distance Ensemble engine automatically picks the best performing algorithm per tissue sample based on unsupervised spatial metrics without requiring ground-truth domain annotations.

---
### 🛠️ How to Run Real Benchmark Experiments
To run real Visium or Slide-seq benchmark datasets (DLPFC, Mouse Brain, Breast Cancer):
```bash
# Execute ACT / FACT / SCATTER from main entry point
python main.py

# Or evaluate multi-method ensemble selection
python Ensemble.py
```
